# Table S14 (tabG02): the v7 matched-comparator estimate by assembly quality (GTDB r232, D15)

**Status:** draft, Gadi. Table S14 in the `STYLE.md` numbering (#137).

## Provenance

| data file | built from | producing stage |
|---|---|---|
| `data/tabG02_assembly_strata_v7.tsv.gz` | `Validation/gtdb/matched_comparator/assembly_strata_v7_{bac,arc}.tsv` in this tree (regenerated with #136's code; the same files go to `main` in #138, from `4e08030`) | `Validation/gtdb/scripts/43_assembly_strata_v7.py` (#134, #136), PBS 181387980 |

- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --repo <this checkout> --only tabG02`, a verbatim copy of both domains and both stratum schemes. It reproduces `data/tabG02_assembly_strata_v7.tsv.gz` byte for byte from this tree.
- **Method** (`Validation/gtdb/assembly_strata_v7.md`):
  - It computes the `41_comparator_summary.py` quantities within contig-N50 and CheckM2-completeness quartiles. The quartile edges are those of `supplementary_analyses.md` §7, from pooled v6 query rows.
  - The comparator is weighted by draw multiplicity.
  - The CIs are first-order 95% intervals.
  - An unstratified row reproduces all 54 published metrics.
- **This table shows scheme (i):** strata by the candidate's genome, so each matched pair stays together. Scheme (ii), with each ORF placed by its own genome, is in the data file as a sensitivity analysis.
- **Prodigal share:** the report's larger-background value. The matched comparator is the larger background in all 32 strata, which the notebook asserts.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

d = pd.read_csv(DATA / "tabG02_assembly_strata_v7.tsv.gz", sep="\t")
d = d[d.scheme.isin(["candidate_genome", "own_genome"])]
COV = {"n50_contigs": "contig N50", "checkm2_completeness": "CheckM2 completeness"}
SHARES = [("share_afdb_swissprot", "Swiss-Prot"), ("share_pdb100", "PDB100"),
          ("share_cath50", "CATH50"), ("share_any_sp_pdb_cath", "any of three")]

# The comparator is the larger Prodigal background in every stratum (#136).
w = d.pivot_table(index=["domain_name", "scheme", "covariate", "stratum"],
                  columns="metric", values="value")
assert (w.prodigal_comparator >= w.prodigal_intergenic_lo).all()
assert np.allclose(w.prodigal_share_larger_bg, w.prodigal_share_clean_bg)


def rng(lo, hi, cov):
    f = (lambda x: f"{x / 1000:.1f} kb") if cov == "n50_contigs" else (lambda x: f"{x:.2f}%")
    if np.isinf(lo):
        return f"< {f(hi)}"
    if np.isinf(hi):
        return f"≥ {f(lo)}"
    return f"{f(lo)} – {f(hi)}"


rows, plain = [], []
p = d[d.scheme == "candidate_genome"]
for dom in ("bacteria", "archaea"):
    for cov in ("n50_contigs", "checkm2_completeness"):
        for q in ("Q1", "Q2", "Q3", "Q4"):
            g = p[(p.domain_name == dom) & (p.covariate == cov) & (p.stratum == q)]
            m = {r.metric: (r.value, r.lo, r.hi) for r in g.itertuples()}
            r = {"domain": dom, "covariate": COV[cov], "quartile": q,
                 "range": rng(g.cov_lo.iloc[0], g.cov_hi.iloc[0], cov),
                 "candidates": int(m["n_candidate"][0])}
            pr = dict(r)
            for k, lab in SHARES:
                v, lo, hi = (x * 100 for x in m[k])
                r[f"{lab} share, % (95% CI)"] = f"{v:.1f} ({lo:.1f}–{hi:.1f})"
                pr[f"{k}"], pr[f"{k}_lo"], pr[f"{k}_hi"] = v, lo, hi
            v, lo, hi = (x * 100 for x in m["prodigal_share_clean_bg"])
            r["Prodigal share, % (95% CI)"] = f"{v:.1f} ({lo:.1f}–{hi:.1f})"
            pr["prodigal_share"], pr["prodigal_share_lo"], pr["prodigal_share_hi"] = v, lo, hi
            for k, lab in (("ladder_C3_excess", "C3 excess"), ("ladder_C7_excess", "C7 excess")):
                r[lab] = f"{m[k][0]:,.0f}"
                pr[k] = m[k][0]
            r["with-replacement draws"] = int(m["n_comparator_with_replacement_draws"][0])
            pr["n_comparator_with_replacement_draws"] = r["with-replacement draws"]
            rows.append(r)
            plain.append(pr)
t = pd.DataFrame(rows)
pt = pd.DataFrame(plain)
pt.to_csv(IMAGES / "tabG02_assembly_strata.tsv", sep="\t", index=False, float_format="%.6f")

md = ["| " + " | ".join(t.columns) + " |", "|" + "|".join("---" for _ in t.columns) + "|"]
md += ["| " + " | ".join(str(x) for x in r) + " |" for r in t.itertuples(index=False)]
(IMAGES / "tabG02_assembly_strata.md").write_text(
    "**Table S14.** v7 matched-comparator estimate by assembly-quality quartile "
    "(GTDB r232; strata by candidate genome).\n\n" + "\n".join(md) + "\n")
print("wrote", IMAGES / "tabG02_assembly_strata.tsv", IMAGES / "tabG02_assembly_strata.md")


In [ ]:
# Numbers the Results text quotes (G-17), each to within its rounding; a changed input fails here.
def val(dom, cov, q, k):
    return float(pt[(pt.domain == dom) & (pt.covariate == COV[cov]) & (pt.quartile == q)][k].iloc[0])


def quoted(x, target):
    assert abs(x - target) <= 0.05 + 1e-9, (x, target)


quoted(val("bacteria", "n50_contigs", "Q1", "share_any_sp_pdb_cath"), 17.4)
quoted(val("bacteria", "n50_contigs", "Q4", "share_any_sp_pdb_cath"), 10.3)
quoted(val("bacteria", "checkm2_completeness", "Q1", "share_any_sp_pdb_cath"), 19.9)
quoted(val("bacteria", "checkm2_completeness", "Q4", "share_any_sp_pdb_cath"), 9.0)
quoted(val("bacteria", "n50_contigs", "Q1", "prodigal_share"), 44.8)
quoted(val("bacteria", "n50_contigs", "Q4", "prodigal_share"), 31.1)
quoted(val("archaea", "n50_contigs", "Q1", "prodigal_share"), 47.0)
quoted(val("archaea", "n50_contigs", "Q4", "prodigal_share"), 32.6)
lows = pt[[c for c in pt.columns if c.startswith("share_") and c.endswith("_lo")]].to_numpy()
quoted(lows.min(), 2.97)
assert lows.min() > 2.9 and (pt.ladder_C3_excess > 0).all() and (pt.ladder_C7_excess > 0).all()
print(f"checks pass; smallest lower CI bound {lows.min():.3f}%")
print(t.to_string(index=False))


## Caption (draft)

**Table S14. The v7 matched-comparator estimate by assembly-quality quartile (GTDB r232).**

For each quartile of contig N50 and of CheckM2 completeness (the edges of the pre-matching analysis), the table gives:
- the number of candidates;
- implied real-gene shares, (candidate − comparator) / (annotated − comparator), on full-length structural hits in Swiss-Prot, PDB100, CATH50 and any of the three, with 95% CIs;
- the C3 and C7 excess over the scaled comparator;
- the Prodigal share;
- the number of comparator draws made with replacement.

Strata follow the candidate's genome, so each candidate and its matched comparator stay together (decision D15). The comparator is weighted by draw multiplicity. The excess is positive in every stratum: every lower CI bound is above 2.9% (smallest 2.97%, bacterial CATH50 in the top completeness quartile). In bacteria the combined share and the Prodigal share are higher in poorer assemblies. The sensitivity analysis, which places each ORF by its own genome, is in `data/tabG02_assembly_strata_v7.tsv.gz` and `Validation/gtdb/assembly_strata_v7.md`.
